In [1]:
%cd ../../..

/home/bhkuser/bhklab/katy/ispy2-r2r


In [2]:
from damply import dirs
import pandas as pd
from pathlib import Path
import yaml
from readii.process.subset import getOnlyPyradiomicsFeatures

/home/bhkuser/bhklab/katy/ispy2-r2r/.pixi/envs/default/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
clinical = pd.read_excel(dirs.RAWDATA / "TCIA_ISPY2/clinical/ISPY2-Imaging-Cohort-1-Clinical-Data.xlsx")
clinical = clinical.sort_values(by='Patient_ID', ignore_index=True)
clinical['Patient_ID'] = clinical['Patient_ID'].astype('str')

feats_ISPY = pd.read_csv(dirs.PROCDATA / "TCIA_ISPY2/features/pyradiomics/v1_TCIA_ISPY2_feature_extracted.csv")
feats_ACRIN = pd.read_csv(dirs.PROCDATA / "TCIA_ACRIN-6698/features/pyradiomics/v1_TCIA_ACRIN-6698_feature_extracted.csv")

2026-02-17 | WARNING | Environment variable 'RAWDATA' is not set. Using default path relative to project root.
2026-02-17 | WARNING | Environment variable 'PROCDATA' is not set. Using default path relative to project root.


In [4]:
radiomics = pd.concat([feats_ISPY, feats_ACRIN])

radiomics = radiomics.rename(columns={'PatientTimepoint': 'Sample_ID'})

radiomics.sort_values(by='Sample_ID', ascending=False)

# Split up the SampleID from R2R into the PatientID and SampleNumber
split_ids = radiomics['Sample_ID'].str.split('_', expand = True)

# Split the PatientID into the dataset name and numeric pat ID
# Use rsplit to handle ACRIN-6698 dataset name
split_patids = split_ids[0].str.rsplit('-', n=1, expand=True)
# Insert these columns into the radiomics dataframe
try:
    radiomics.insert(1, column='Dataset', value=split_patids[0], allow_duplicates=False)
except ValueError:
    print('Dataset already in radiomics dataframe.')

try:
    radiomics.insert(2, column='Patient_ID', value=split_patids[1], allow_duplicates=False)
except ValueError:
    print('PatientID already in radiomics dataframe.')

try:
    radiomics.insert(3, column='Sample_Number', value=split_ids[1], allow_duplicates=False)
except ValueError:
    print('SampleNumber already in radiomics dataframe.')


In [5]:
radiomics_t0 = radiomics.groupby(by='Patient_ID').head(1).reset_index(drop=True)

radiomics_t1 = radiomics.groupby(by='Patient_ID').nth(2).reset_index(drop=True)

In [6]:
radiomics_t0.columns

Index(['Sample_ID', 'Dataset', 'Patient_ID', 'Sample_Number', 'ID', 'Image',
       'Mask', 'diagnostics_Versions_PyRadiomics',
       'diagnostics_Versions_Numpy', 'diagnostics_Versions_SimpleITK',
       ...
       'lbp-2D_gldm_LargeDependenceLowGrayLevelEmphasis',
       'lbp-2D_gldm_LowGrayLevelEmphasis',
       'lbp-2D_gldm_SmallDependenceEmphasis',
       'lbp-2D_gldm_SmallDependenceHighGrayLevelEmphasis',
       'lbp-2D_gldm_SmallDependenceLowGrayLevelEmphasis',
       'lbp-2D_ngtdm_Busyness', 'lbp-2D_ngtdm_Coarseness',
       'lbp-2D_ngtdm_Complexity', 'lbp-2D_ngtdm_Contrast',
       'lbp-2D_ngtdm_Strength'],
      dtype='object', length=1455)

# Set Patient ID as index of all dataframes

In [6]:
radiomics_t0.set_index('Patient_ID', inplace=True)
radiomics_t1.set_index('Patient_ID', inplace=True)

clinical.set_index('Patient_ID', inplace=True)

# Save out clinical and radiomic data

In [ ]:
out_path = dirs.PROCDATA / "TCIA_ISPY2" / "BRADCURE_analysis"
out_path.mkdir(parents=True, exist_ok=True)
clinical.to_csv(out_path / "clinical_ISPY2_combined.csv", index_label="Patient_ID")
radiomics_t0.to_csv(out_path / "radiomics_ISPY2_combined.csv", index_label="Patient_ID")



In [48]:
features_t0 = getOnlyPyradiomicsFeatures(radiomics_t0)
features_t0.to_csv(out_path / "features_only_ISPY2_combined.csv", index_label="Patient_ID")

# Data Splitting

In [ ]:
from sklearn.model_selection import train_test_split

train_pats, test_pats = train_test_split(clinical.index,
                            test_size=0.2, 
                            random_state=10, 
                            shuffle=True)

train_pats = train_pats.sort_values()
test_pats = test_pats.sort_values()

In [25]:
tr_clinical = clinical.loc[train_pats]
tr_radiomics_t0 = radiomics_t0.loc[train_pats]

test_clinical = clinical.loc[test_pats]
test_radiomics_t0 = radiomics_t0.loc[test_pats]


# AIM 1: Virtual biopsy for key pathologic features and moelecular assay scores

Outcomes: HR + HER2

In [39]:
tr_aim1_clinical_t0 = tr_clinical[['HR', 'HER2']]
test_aim1_clinical_t0 = test_clinical[['HR', 'HER2']]

In [ ]:


tr_features_t0 = getOnlyPyradiomicsFeatures(tr_radiomics_t0)
test_features_t0 = getOnlyPyradiomicsFeatures(test_radiomics_t0)

## Luo HER2 status SVM model

In [31]:
def load_signature(signature_name) -> dict:
    signature_dir = dirs.CONFIG / "signatures"
    signature_path = signature_dir / f"{signature_name}.yaml"

    try:
        with signature_path.open('r', encoding='utf-8') as f:
            yaml_data = yaml.safe_load(f)
            if not isinstance(yaml_data, dict):
                message = "ROI match YAML must contain a dictionary"
                raise TypeError(message)
    except Exception:
        message = f"Error loading signature YAML file at {signature_path}"
        raise IOError(message)
    
    return pd.Series(yaml_data['signature'])

In [33]:
signature_name = "luo_2024_her2_pyradiomics"

signature_features = load_signature(signature_name)

feature_names = signature_features.index.to_list()

tr_sig_features_t0 = tr_features_t0[feature_names]
test_sig_features_t0 = test_features_t0[feature_names]

In [34]:
from sklearn.ensemble import RandomForestClassifier

luo_clf = RandomForestClassifier(random_state=0)
luo_clf.fit(tr_sig_features_t0, tr_aim1_clinical_t0['HER2'])



,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [37]:
tr_predict = luo_clf.score(tr_sig_features_t0, tr_aim1_clinical_t0['HER2'])
tr_predict

1.0

In [41]:
luo_clf.score(test_sig_features_t0, test_aim1_clinical_t0['HER2'])

0.6954314720812182